# 05 · 特徵工程：把原始欄位變成模型的養分

模型不會自己「看懂」原始資料。**特徵工程**就是把原始欄位加工成模型更好吃的形式——這往往比換更厲害的模型還有效。

三件最常見的事:
1. **類別編碼**:`sex="female"` 這種文字,模型看不懂,要轉成數字。
2. **衍生特徵**:從現有欄位算出新的、更有意義的欄位(例如家庭人數)。
3. **數值縮放**:把不同尺度的數值拉到同一範圍。

## 1. 載入乾淨資料

In [ ]:
%pip install -q -U seaborn scikit-learn

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"])
print("乾淨資料:", df.shape)

## 2. 類別編碼

二元類別(sex)直接映射 0/1;多類別(embarked)用 one-hot 展開成多欄。

In [ ]:
feat = df.copy()
feat["sex"] = feat["sex"].map({"male": 0, "female": 1})        # 二元 → 0/1
feat = pd.get_dummies(feat, columns=["embarked"], prefix="emb")  # one-hot
print("編碼後新增的欄位:", [c for c in feat.columns if c.startswith("emb_")])

## 3. 衍生特徵:創造更有意義的欄位

`sibsp`(兄弟姊妹/配偶)+ `parch`(父母/子女)+ 自己 = **家庭人數**;再衍生一個「是否獨自一人」。這種特徵常比原始欄位更能預測生還。

In [ ]:
feat["family_size"] = feat["sibsp"] + feat["parch"] + 1
feat["is_alone"] = (feat["family_size"] == 1).astype(int)
print(feat.groupby("is_alone")["survived"].mean())
# 獨自一人的生還率明顯較低——新特徵抓到了原始欄位沒直接表達的訊號。

## 4. 數值縮放

`age` 跨 0–80、`fare` 跨 0–500,尺度差很多。`StandardScaler` 把它們標準化到同一基準。

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
feat[["age", "fare"]] = scaler.fit_transform(feat[["age", "fare"]])
print(feat[["age", "fare"]].describe().round(2).loc[["mean", "std"]])
# 標準化後:平均 ~0、標準差 ~1。許多模型(尤其線性模型)吃這個更穩。

## 小結

- **特徵工程三招**:類別編碼(map / one-hot)、衍生特徵、數值縮放。
- **好特徵勝過好模型**:`family_size` / `is_alone` 抓到了原始欄位沒直接講的訊號。
- 縮放讓不同尺度的數值公平比較,線性模型特別需要。

下一課:用**統計檢定**確認「這些差異是真的,還是運氣」。